## Gold Layer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

### Read Silver Table

In [0]:
df_silver = spark.read.table("fintech.silver.stock_prices")

### Update dim_date
- For dim_date we create date_key as a surrogate key in more deterministic way by using date_format(YYYYMMDD)

In [0]:
# =========================================================
# Get distinct dates from Silver
# =========================================================
df_dates = (
    df_silver
    .select("trade_date")
    .distinct()
    .filter(
        F.col("trade_date").isNotNull()
    )
)

# =========================================================
# Check if dim_date exists
# =========================================================
dim_date_exists = spark.catalog.tableExists(
    "fintech.gold.dim_date"
)

# =========================================================
# Create dimension if it does not exist
# =========================================================
if not dim_date_exists:

    df_dim_date = (
        df_dates
        .withColumn(
            "date_key",
            F.date_format(
                "trade_date",
                "yyyyMMdd"
            ).cast("int")
        )
        .withColumn(
            "full_date",
            F.col("trade_date")
        )
        .withColumn(
            "year",
            F.year("trade_date")
        )
        .withColumn(
            "quarter",
            F.quarter("trade_date")
        )
        .withColumn(
            "month",
            F.month("trade_date")
        )
        .withColumn(
            "month_name",
            F.date_format(
                "trade_date",
                "MMMM"
            )
        )
        .withColumn(
            "week_of_year",
            F.weekofyear("trade_date")
        )
        .withColumn(
            "day",
            F.dayofmonth("trade_date")
        )
        .withColumn(
            "day_of_week",
            F.dayofweek("trade_date")
        )
        .withColumn(
            "day_name",
            F.date_format(
                "trade_date",
                "EEEE"
            )
        )
        .withColumn(
            "is_weekend",
            F.dayofweek("trade_date").isin(1, 7)
        )
        .select(
            "date_key",
            "full_date",
            "year",
            "quarter",
            "month",
            "month_name",
            "week_of_year",
            "day",
            "day_of_week",
            "day_name",
            "is_weekend"
        )
    )

    (
        df_dim_date
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("fintech.gold.dim_date")
    )

# =========================================================
# Existing dimension
# =========================================================
else:
    df_dim_date = (
        spark.table("fintech.gold.dim_date")
    )

    # =====================================================
    # Find new dates
    # =====================================================
    df_new_dates = (
        df_dates
        .join(
            df_dim_date.select("full_date"),
            on=df_dates.trade_date == df_dim_date.full_date,
            how="left_anti"
        )
        .select("trade_date")
    )

    # =====================================================
    # Generate attributes for new dates
    # =====================================================
    df_new_dates = (
        df_new_dates
        .withColumn(
            "date_key",
            F.date_format(
                "trade_date",
                "yyyyMMdd"
            ).cast("int")
        )
        .withColumn(
            "full_date",
            F.col("trade_date")
        )
        .withColumn(
            "year",
            F.year("trade_date")
        )
        .withColumn(
            "quarter",
            F.quarter("trade_date")
        )
        .withColumn(
            "month",
            F.month("trade_date")
        )
        .withColumn(
            "month_name",
            F.date_format(
                "trade_date",
                "MMMM"
            )
        )
        .withColumn(
            "week_of_year",
            F.weekofyear("trade_date")
        )
        .withColumn(
            "day",
            F.dayofmonth("trade_date")
        )
        .withColumn(
            "day_of_week",
            F.dayofweek("trade_date")
        )
        .withColumn(
            "day_name",
            F.date_format(
                "trade_date",
                "EEEE"
            )
        )
        .withColumn(
            "is_weekend",
            F.dayofweek("trade_date").isin(1, 7)
        )
        .select(
            "date_key",
            "full_date",
            "year",
            "quarter",
            "month",
            "month_name",
            "week_of_year",
            "day",
            "day_of_week",
            "day_name",
            "is_weekend"
        )
    )
    # =====================================================
    # Insert new dates
    # =====================================================
    if not df_new_dates.isEmpty():
        (
            df_new_dates
            .write
            .format("delta")
            .mode("append")
            .saveAsTable("fintech.gold.dim_date")
        )